# Folder Layout

In [7]:
import os
os.makedirs("src", exist_ok=True)

In [ ]:
from pathlib import Path
folder = Path("src")
files = [
    "__init__.py",
    "tilecoding.py",    # the function approximator
    "exploration.py",   # the nine exploration rules
    "agents.py",        # SARSA (lambda) & Q-Learning
    "runner.py",        # one training run 
    "sweep.py",         # many runs, in parallel, resumable
    "analysis.py",      # statistics
    "plots.py"          # figures
]
sub_folders = [
    "scripts",          
    "results",          # .pkl files 
    "figures",          # .png files
    "paper"             # .tex files
]
f_folder = Path("src/scripts")
script_files = [
    "tune.py",          # phase 1: choose hyperparameters
    "evaluate.py",      # phase 2: the real experiment
    "make_figures.py"   # phase 3: figures and tables
]

for file in files:
    (folder/file).touch(exist_ok=True)

for sub_folder in sub_folders:
    (folder/sub_folder).mkdir(exist_ok=True)

for script_file in script_files:
    (f_folder/script_file).touch(exist_ok=True)

# Building Gymnasium Wrapper

In [8]:
%%writefile src/runner.py

import numpy as np
import gymnasium as gym

class GymEnv:
    """Adapts Gymnasium to a 4-tuple step interface"""

    def __init__(self, env_id, seed = None):                            # A function that is run on every class initialization, self is the object being created
        self.env = gym.make(env_id)                                     # Creates the environment specified by the call 
        self.spec_name = env_id                                         # name of the environment (separated for convenience)
        self.n_actions = int(self.env.action_space.n)                   # number of actions in the environment
        ms = self.env.spec.max_episode_steps                                 # maximum number of steps in an episode
        self.max_episode_steps = int(ms) if ms is not None else 10_000  # gets the maximum number of steps in an episode, if there is none, sets it to 10,000
        self._next_seed = seed
    
    def reset(self):
        """Resets the environment and returns the initial state"""
        obs, _ = self.env.reset(seed=self._next_seed)                   # resets the environment and returns the initial observation
        self._next_seed = None              # seeds once, then follows the stream; the whole experiment is reproducible, but episodes have random initial states
        return np.asarray(obs, dtype=np.float64) # returns the initial observation as a numpy array of type float64

    def step(self, action):
        """Takes a step and returns observation, reward, termination status, truncation status"""
        obs, r, term, trunc, _ = self.env.step(int(action))     # action is typecasted to python int, in case it is a numpy int, which gymnasium does not accept
        return (np.asarray(obs, dtype=np.float64), float(r), bool(term), bool(trunc))

Overwriting src/runner.py


# Tile Coder

## The IHT Class
It does not do tile coding itself, rather it manages the mapping of a *which tile it hit* to *where in the weight array represents the tile*.

IHT = Index Hash Table

The object will remember:

* the maximum number of available feature indices,
* which tiles have already been seen,
* which index each tile received,
* how often the table overflowed.

\_\_slots\_\_: This is a Python **feature** which says that objects of the class can only have the attributes passed as its argument. This also helps to reduce memory overhead (as only the passed arguments can be attributed) and protect from naming errors too. 

\_d: Python dictionary, initiated as empty inside \_\_init()\_\_. This will store the tile key and allocated integer index pair. The `\_' in a variable name is a convention to indicate that the variable is not truly private; it can be accessed but should not need to, by other parts of the program. 

\_\_len(self)\_\_: It enables using len() to get the size of \_d, without having to explicitly address it. For instance, say, iht = IHT(4096), and there have been 3 allocations to `\_d'. Then, len(iht) = 3. 

@property: It enables to use the function following it to be used as an attribute. So, fullness can be used as iht.fullness instead of iht.fullness().

fullness(self): This calculates the fraction of the IHT capacity already occupied. 

get_index(): This function returns the weight index associated with the tile indicated by `key'. The workflow of the function can be summarized as: 

```
Have I seen this tile before?
            │
       ┌────┴────┐
      yes        no
       │          │
return old       Is table full?
index             │
             ┌────┴────┐
            no         yes
             │           │
      allocate new    collision/
         index          modulo
```

d.get(): The get() is an useful function for Python dictionaries. It fetches the other attribute's value associated with its argument from the dictionary. For example, let d = {
    2: 5
    3: 29
    7: 11
}
Then d.get(3) returns 29. However, d.get(1) would return `None' as the key 1 is not present in d. 


In [4]:
%%writefile src/tilecoding.py

class IHT:
    """Index Hash Table: maps tile coordinates to dense indices in [0, size).

    Collision-free until `size` distinct tiles have been seen; afterwards
    new tiles wrap by modulo.
    """

    __slots__ = ("size", "_d", "overfull_count")

    def __init__(self, size):
        self.size = int(size)
        self._d = {}
        self.overfull_count = 0

    def __len__(self):
        return len(self._d)

    @property
    def fullness(self):
        return len(self._d) / self.size

    def get_index(self, key):
        d = self._d
        idx = d.get(key)

        if idx is not None:
            return idx

        n = len(d)

        if n >= self.size:
            self.overfull_count += 1
            return key % self.size

        d[key] = n
        return n

Overwriting src/tilecoding.py


## The Tile Coder

This particular code does not yet encode a state. It is the constructor: it prepares the geometry, scales, offsets, and hash table that the later methods will use.

```class TileCoder:``` An object of this class will know:
* the state-space boundaries,
* the number of state dimensions,
* how many tilings exist,
* how many tiles exist per dimension,
* how continuous values should be scaled,
* how each tiling is offset,
* which IHT maps tile coordinates to feature indices.

memory_size: It is the size of the IHT (the maximum dictionary (_d) size). 

low & high: These are the lower & higher bounds of the dimensions. For instance, in case of Mountaincar, low = [-1.2, 0.07] which is the lowest possible value of position & velocity respectively. `self.low.shape[0]' gives the number of dimensions of the states. 

if np.isscalar()...else: This check enables either assigning the same number of tiles per dimension (if condition) or varying number (else condition).

np.full(): This has the syntax ```np.full(size, value)```. It creates an np array of size ```size``` with all elements having the value ```value```. 

span: One thing to note is that this is an array not a scalar for multidimensional states (since low & high are arrays). Now the span[span==0] = 1.0 is a zero span safety. It turns those entries for which span = 0 to 1, leaving the other dimension spans untouched. 

scale: It converts the physical state dimension values into tile coordinate values. It is constructed by elementwise division of tpd & span. This in words mean *how many tiles per unit of dimension is allocated in the tile coordinate corresponding to that dimension*. 

x[None,:]: This turns the preceding 1D array into a 2D row matrix of size (1,n(x)). Similarly, x[:,None] turns it into a column matrix. 

In [5]:
%%writefile -a src/tilecoding.py

class TileCoder:
    def __init__(self, low, high, num_tilings=8, tiles_per_dim=8, memory_size=4096):
        self.low = np.asarray(low, dtype=np.float64)
        self.high = np.asarray(high, dtype=np.float64)
        self.dim = int(self.low.shape[0])
        self.num_tilings = int(num_tilings)

        if np.isscalar(tiles_per_dim):
            tpd = np.full(self.dim, int(tiles_per_dim), dtype=np.int64)
        else:
            tpd = np.asarray(tiles_per_dim, dtype=np.int64)
        
        self.tiles_per_dim = tpd

        span = self.high - self.low
        span[span == 0] = 1.0
        self.scale = tpd / span

        self.iht = IHT(memory_size)
        self.memory_size = int(memory_size)

        t_idx = np.arange(self.num_tilings, dtype=np.int64)
        odd = (1 + 2 * np.arange(self.dim, dtype=np.int64))[None, :]
        self._offsets = t_idx[:, None] * odd


Appending to src/tilecoding.py


## The Hot Function

```indices(state)``` takes one continuous state, finds the one active tile in each tiling, and converts those tiles into integer feature indices using the IHT.

```//```: This is integer floor division. So, 62//8 = 7, not 7.75.

```get = self.iht.get_index```: This creates a local variable copy of self.iht.get_index(). Now simply get(key) can be used. Also, since this will be called millions of time during training, this copy helps python to just resolve it once instead of calling it each time. 

```*```: The \* in \*coords[t].tolist() means take the elements out of this iterable and insert them individually here. So, say t = 2 and coords[t] = array([3, 5]). Then (to, \*coords[t].tolist()) becomes (2, \*[3, 5]) = (2, 3, 5).

```hash()```: The hash() function converts its argument tuple into a Python hash integer. This conversion to *integer* is necessary as return key % self.size only works if key is an integer. 

In [1]:
%%writefile -a src/tilecoding.py

    def indices(self, state):
        """Return the `num_tilings` active feature indices for `state`"""

        s = np.clip(np.asarray(state, dtype=np.float64), self.low, self.high)
        q = np.floor((s-self.low) * self.scale * self.num_tilings).astype(np.int64)

        coords = (q[None, :] + self._offsets) // self.num_tilings

        out = np.empty(self.num_tilings, dtype=np.int64)
        get = self.iht.get_index
        for t in range(self.num_tilings):
            out[t] = get(hash((t, *coords[t].tolist())))
        return out

Appending to src/tilecoding.py


## The Linear Action-Value Function

This class is the part that turns the active tile indices from TileCoder.indices(state) into actual action-value estimates $\hat{q}(s, a)$ using linear function approximation. 

```sum(axis=1)```: This sums across columns of the 2D array. 

In [ ]:
%%writefile -a src/tilecoding.py

class LinearQ:
    """Q(s,a) = sum of w[a, i] over the active features of s (state)"""

    __slots__ = ("w", "n_actions", "n_features")

    def __init__(self, n_actions, n_features, init=0.0):
        self.n_actions = int(n_actions)
        self.n_features = int(n_features)
        self.w = np.full((n_actions, n_features), float(init), dtype=np.float64)

    def q_all(self, idx):
        """Q(s, .) for every action -> shape (n_actions, )"""
        return self.w[:, idx].sum(axis=1)

## The Configuration Dictionary

```cfg```: This is a nested dictionary. So, return dict(cfg[env_id]) returns a dictionary associated with the passed environment name. 

In [2]:
%%writefile -a src/tilecoding.py

def default_tiling_config(env_id):
    cfg = {
        "MountainCar-v0": dict(num_tilings=8, tiles_per_dim=8, memory_size=4096),
        "CartPole-v1": dict(num_tilings=8, tiles_per_dim=6, memory_size=32768),
        "Acrobot-v1": dict(num_tilings=8, tiles_per_dim=6, memory_size=131072),
        "LunarLander-v3": dict(num_tilings=16, tiles_per_dim=4, memory_size=262144),
    }

    return dict(cfg[env_id])

Appending to src/tilecoding.py


## Quick Check

Same states have identical indices

In [4]:
import src.tilecoding as TC

tc = TC.TileCoder([-1.2, -0.07], [0.6, 0.07], 8, 8, 4096)

i1 = tc.indices([-0.5, 0.01])
i2 = tc.indices([-0.5, 0.01])

assert len(i1) == 8
assert (i1==i2).all()
assert (i1>=0).all() and (i1<4096).all()
print(i1)

[0 1 2 3 4 5 6 7]


Nearby states have some common indices

In [ ]:
import src.tilecoding as TC

tc = TC.TileCoder([-1.2, -0.07], [0.6, 0.07], 8, 8, 4096)

i1 = tc.indices([-0.5, 0.01])
i2 = tc.indices([-0.49, 0.011])

assert len(i1) == 8
assert (i1==i2).any()
assert (i1!=i2).any()

assert (i1>=0).all() and (i1<4096).all()
assert (i2>=0).all() and (i2<4096).all()

print(i1)
print(i2)


[0 1 2 3 4 5 6 7]
[0 8 2 3 4 5 6 9]


Distant states have no common indices

In [8]:
import src.tilecoding as TC

tc = TC.TileCoder([-1.2, -0.07], [0.6, 0.07], 8, 8, 4096)

i1 = tc.indices([-0.5, 0.01])
i2 = tc.indices([-0.9, -0.06])

assert len(i1) == 8
assert (i1!=i2).all()

assert (i1>=0).all() and (i1<4096).all()
assert (i2>=0).all() and (i2<4096).all()

print(i1)
print(i2)


[0 1 2 3 4 5 6 7]
[ 8  9 10 11 12 13 14 15]


# Exploration

## The Explorer

It is the common base/interface that all nine exploration rules will follow.The main idea is that **the agent asks the explorer for an action, instead of it knowing which exploration strategy it is following**. This enables the agent to remain identical, while only the exploration object is swapped. Every explorer exposes the same interface.
```
Agent
  │
  │ asks for action
  ▼
Explorer
  │
  ├── DecayExplorer
  ├── VDBEExplorer
  ├── BoltzmannExplorer
  ├── ProposedExplorer
  ├── TileUCBExplorer
  └── ...
```
```name``` is a class attribute, while ```self.t``` is an instance attribute, associated with an individual object. 

```def _epsilon_now(self, feat_idx=None)```: This defines the rule for selecting the epsilon to be used now. 

```raise NotImplementedError```: The base Explorer does not know how epsilon should be calculated.Different  subclasses have different answers. Therefore the base class deliberately says: *I cannot implement this. A subclass must tell me what _epsilon_now() means.*

```rng.random()``` produces a random number uniformly in [0,1). 

Till now conceptually, one step looks like:
```
Current state s
       │
       ▼
TileCoder.indices(s)
       │
       ▼
LinearQ.q_all(idx)
       │
       ▼
[Q(s,0), Q(s,1), Q(s,2)]
       │
       ▼
Explorer.select(...)
       │
       ▼
choose action a
       │
       ▼
Environment.step(a)
       │
       ▼
reward r, next state s'
       │
       ▼
compute TD error δ
       │
       ├──────────────► update LinearQ weights
       │
       └──────────────► explorer.update(δ, ...)
```

Summary of this class:
```
| Part              | Purpose                                             |
| ----------------- | --------------------------------------------------- |
| `name`            | identifies the strategy                             |
| `uses_features`   | says whether tile features are required             |
| `n_actions`       | number of available actions                         |
| `t`               | number of action-selection calls                    |
| `current_epsilon` | most recently used \(\epsilon\)                     |
| `_epsilon_now()`  | compute current exploration level                   |
| `update()`        | receive TD/value/feature information after learning |
| `reset_episode()` | reset strategy-specific episode state               |
| `select()`        | perform epsilon-greedy action selection             |
```

## The Tie-breaker

Its job is:

Find the action(s) having the largest Q-value. If there is only one, choose it. If several actions are tied for maximum, randomly choose one of those tied actions.

```q == m```: This performs element by element comparison through an array. Since q is an array and m is the maximum value, each element in q is matched withm. The resulting array ties will have *False* where matches were not found, and *True* elsewhere. 

```np.flatnonzero()``` returns the indices where the input is nonzero/True.

In [9]:
%%writefile src/exploration.py

import numpy as np

def argmax_random_tie(q, rng):
    """Argmax with ties broken uniformly at random"""

    m = q.max()
    ties = np.flatnonzero(q == m)
    if ties.size == 1:
        return int(ties[0])
    return int(ties[rng.integers(ties.size)])

Overwriting src/exploration.py


In [10]:
%%writefile -a src/exploration.py

class Explorer:
    name = "base"
    uses_features = False

    def __init__(self, n_actions):
        self.n_actions = int(n_actions)
        self.t = 0
        self.current_epsilon = 0.0

    def _epsilon_now(self, feat_idx=None):
        raise NotImplementedError
    
    def update(self, td_error, value=0.0, feat_idx=None):
        return None
    
    def reset_episode(self):
        return None
    
    def select(self, q_values, rng, feat_idx=None):
        eps = self._epsilon_now(feat_idx)
        self.current_epsilon = eps
        self.t += 1
        if rng.random() < eps:
            return int(rng.integers(self.n_actions))
        return argmax_random_tie(q_values, rng)

Appending to src/exploration.py


## Schedules


### Fixed $\epsilon$

```super().__init__(n_actions)```: As FixedEpsilon has its own ```__init__()```, ```super().__init__()``` is needed to tell Python to run its parent's one (*Explorer's ```__init__()```*). The flow summarized:
```
FixedEpsilon.__init__()
        ↓
super().__init__(3)
        ↓
Explorer.__init__(3)
        ↓
self.n_actions = 3
self.t = 0
self.current_epsilon = 0.0
```

In [1]:
%%writefile -a src/exploration.py

class FixedEpsilon(Explorer):
    name = "fixed"
    
    def __init__(self, n_actions, epsilon=0.1):
        super().__init__(n_actions)
        self.epsilon = float(epsilon)

    def _epsilon_now(self, feat_idx=None):
        return self.epsilon

Appending to src/exploration.py


### Decay $\epsilon$

The $\epsilon$ decays by the formula: 
\begin{equation}
    \epsilon_{end} = \epsilon_{start}\,e^{rt}
\end{equation}

The max() in the log and numerator are protection mechanisms against unwanted zeros. 

In [2]:
%%writefile -a src/exploration.py

class DecayEpsilon(Explorer):
    name = "decay"

    def __init__(self, n_actions, eps_start=1, eps_end=0.01, decay_steps=50_000, mode="exponential"):
        super().__init__(n_actions)
        self.eps_start, self.eps_end = float(eps_start), float(eps_end)
        self.decay_steps, self.mode = int(decay_steps), mode
        self._rate = np.log(max(eps_end, 1e-12)/eps_start)/max(decay_steps, 1)

Appending to src/exploration.py


### Boltzmann

Boltzmann exploration does not make the explore/exploit split. Instead, every action gets a probability based on its Q-value:
\begin{equation}
    P(a|s) = \cfrac{e^{\frac{Q(s,a)-Q_{max}(s,a)}{\tau}}}{\sum_b e^{\frac{Q(s,b)-Q_{max}(s,b)}{\tau}}}
\end{equation}

It then selects the non-greedy actions by probability $1-P(\text{greedy})$. 

```choice()``` selects the action number according to the probabilities specified by `p'.


In [1]:
%%writefile -a src/exploration.py

class Boltzmann(Explorer):
    name = "boltzmann"

    def __init__(self, n_actions, tau_start=1.0, tau_end=0.05, decay_steps=50_000):
        super().__init__(n_actions)
        self.tau_start, self.tau_end = float(tau_start), float(tau_end)
        self.decay_steps = int(decay_steps)
        self._rate = np.log(max(tau_end, 1e-12)/tau_start)/max(decay_steps, 1)

    def select(self, q_values, rng, feat_idx=None):
        tau = float(max(self.tau_end, self.tau_start * np.exp(self._rate * self.t)))
        self.t += 1
        z = np.clip((q_values - q_values.max()) / max(tau, 1e-8), -50.0, 0.0)
        p = np.exp(z)
        p /= p.sum()
        self.current_epsilon = float(1.0 -p[int(np.argmax(q_values))])
        return int(rng.choice(self.n_actions, p=p))

Appending to src/exploration.py


### VDBE

Unlike FixedEpsilon or DecayEpsilon, its exploration probability is adaptive: it changes according to the magnitude of the TD error.

```delta_param```: This is the $\cfrac{1}{|A|}$ in the $\epsilon$ update formula. 

```@staticmethod```: This tells that the following function belongs conceptually to the VDBE class, but does not require an instance (self). It also does not read variables like self.sigma, self.n_actions etc.

```_f(x)```: min(x, 50.0) is used since exp(-50) is already effectively zero, so exp(-x) for large x is overkill. 

Summarized workflow:
```
                 current state
                      │
                      ▼
                   Q values
                      │
                      ▼
             VDBE current epsilon
                      │
                      ▼
           Explorer.select() chooses
                    action
                      │
                      ▼
                 environment
                      │
                      ▼
              reward + next state
                      │
                      ▼
              compute TD error δ
                      │
                      ▼
                VDBE.update()
                      │
          ┌───────────┴───────────┐
          ▼                       ▼
    compute |αδ|/σ             old epsilon
          │                       │
          ▼                       │
        f(x)                      │
          │                       │
          └───────────┬───────────┘
                      ▼
        ε ← d f + (1-d) ε
                      │
                      ▼
          used on next action
```

In [2]:
%%writefile -a src/exploration.py

class VDBE(Explorer):
    name = "vdbe"

    def __init__(self, n_actions, sigma=1.0, eps_init=1.0, alpha_scale=1.0, eps_min=0.0):
        super().__init__(n_actions)
        self.sigma = float(sigma)
        self.eps = float(eps_init)
        self.delta_param = 1.0 / self.n_actions
        self.alpha_scale = float(alpha_scale)
        self.eps_min = float(eps_min)

    def _epsilon_now(self, feat_idx=None):
        return max(self.eps_min, self.eps)

    @staticmethod
    def _f(x):
        e = np.exp(-min(x, 50.0))
        return float((1.0 - e) / (1.0 + e))
    
    def update(self, td_error, value=0.0, feat_idx=None):
        f = self._f(abs(self.alpha_scale * float(td_error)) / max(self.sigma, 1e-12))
        d = self.delta_param
        self.eps = d * f + (1.0 - d) * self.eps

Appending to src/exploration.py


### RATE

RATE compares two moving quantities:

$ m_{\delta} \approx \text{recent average magnitude of TD errors} $

and

$m_v \approx \text{recent average magnitude of the current value estimate}$

Then, $\rho=\cfrac{m_{\delta}}{m_{\delta}+m_v}$

and, $\epsilon=\epsilon_{min} + (\epsilon_{max}-\epsilon_{min})\rho^{\kappa}$

In [3]:
%%writefile -a src/exploration.py

class RATE(Explorer):
    name = "rate"

    def __init__(self, n_actions, eps_min=0.01, eps_max=1.0, beta=0.01, kappa=1.0):
        super().__init__(n_actions)
        self.eps_min, self.eps_max = float(eps_min), float(eps_max)
        self.beta, self.kappa = float(beta), float(kappa)
        self.m_d = 0.0
        self.m_v = 0.0
        self.rho = 1.0

    def _epsilon_now(self, feat_idx=None):
        return self.eps_min + (self.eps_max - self.eps_min) * (self.rho ** self.kappa)
    
    def update(self, td_error, value=0.0, feat_idx=None):
        b = self.beta
        self.m_d += b * (abs(float(td_error)) - self.m_d)
        self.m_v += b * (abs(float(value)) - self.m_v)
        den = self.m_d + self.m_v
        self.rho = (self.m_d / den) if den > 1e-8 else 1.0

Appending to src/exploration.py


### RATEState

This is the per-feature version of RATE, and it is more interesting than the global RATE because it makes exploration depend on which part of the state space the agent is currently in. RATEState adds a separate TD-error statistic for every tile-coded feature.

One thing to note is that the inheritance chain here is different: 
```
Explorer
   ↑
 RATE
   ↑
RATEState
```

Let, ```feat_idx = np.array([13, 82, 205, 377, 612, 901, 1205, 1600])```. Those might be the eight active tile features for the current state. RATEState uses those indices to determine the local exploration level.

We need `\_seen' because, M[i] = 0 is ambiguous. Does it mean: 

This feature has been visited and its error really is approximately zero?

or:

This feature has never been visited and was merely initialized to zero?

Those are completely different situations. So `\_seen' distinguishes them.

```_epsilon_now()```: 
```
feat_idx absent
    ↓
use global RATE

feat_idx provided
    ↓
use feature-specific RATE
```
```seen = self._seen[feat_idx]```: This is a Boolean mask corresponding to the currently active features (e.g. array([True, True, False, True])). Note that `seen' is not the entire `\_seen' array. It only contains the features associated with the current state. If no active features were seen previously, then $\epsilon = \epsilon_{max} = 1$ is set. This makes sense--- 
```
completely unfamiliar region
        ↓
maximum exploration
```

```local```: This is the average stored TD-error magnitude among the currently active, previously seen features. So, it's a local estimate of how uncertain/unsettled learning appears in this part of the state space.

Why ignore unseen features rather than treating them as zero? Because zero would incorrectly mean: *perfect prediction.*

The $m_{\delta}$ in $\rho$ has been replaced by `local' for this case. 

```super().update()```: This calls the RATE's update function. So RATEState still maintains the global statistics too. Why? Because: $m_v$ is still needed for local normalization, global $\rho$ is needed as fallback when no feat_idx is supplied.

```fresh```: This tells us: which currently active features have never been seen before?

Every previously seen active feature is updated according to: 

$M_i \leftarrow M_i + \beta (|\delta| - M_i)$. 

While the fresh features are initialized to $|\delta|$.